# Extra: BBB KNN 变量权重生成

本 notebook 独立于 OmegaBBB.ipynb，不读取或修改 BBB 文件。

目标：用一套内置的个人贷款审批后清算结果测试数据训练 LightGBM 坏账分类模型，并根据特征重要性生成 BBB KNN 欧式距离计算所需的变量权重集。

最终输出：
- 内置清算结果样本预览
- 坏账 / 非坏账分布
- LightGBM 特征重要性表
- BBB KNN 变量权重表
- 可复制的 Python 字典 `bbb_knn_weights`


In [1]:
from __future__ import annotations

from IPython.display import display
import lightgbm as lgb
import numpy as np
import pandas as pd

model_name = 'LightGBM'
RANDOM_STATE = 42
BBB_KNN_VARIABLES = [
    'loan_amount',
    'expected_return_rate',
    'property_value',
    'ltv_ratio',
    'annual_income',
    'overdue_count',
    'employment_years',
    'credit_score',
    'debt_to_income_ratio',
]


In [2]:
def build_clearance_records() -> list[dict[str, object]]:
    records = []
    for idx in range(60):
        credit_score = 620 + (idx * 17) % 160
        overdue_count = [0, 0, 1, 2, 0, 3, 1, 0, 2, 0, 4, 1][idx % 12]
        debt_to_income_ratio = round(0.22 + ((idx * 7) % 45) / 100, 2)
        ltv_ratio = round(0.45 + ((idx * 5) % 40) / 100, 2)
        loan_amount = 220000 + (idx * 93000) % 1480000
        annual_income = 120000 + (idx * 47000) % 360000
        property_value = round(loan_amount / ltv_ratio, 2)
        expected_return_rate = round(0.045 + ((idx * 3) % 35) / 1000, 3)
        employment_years = 1 + (idx * 3) % 15

        risk_points = 0
        risk_points += 2 if credit_score < 670 else 1 if credit_score < 700 else 0
        risk_points += 3 if overdue_count >= 4 else 2 if overdue_count >= 2 else 0
        risk_points += 1 if debt_to_income_ratio > 0.50 else 0
        risk_points += 1 if ltv_ratio > 0.72 else 0
        risk_points += 1 if annual_income < 180000 else 0
        is_bad_debt = int(risk_points >= 4)

        records.append({
            'loan_id': f'PL{idx + 1:04d}',
            'loan_amount': float(loan_amount),
            'expected_return_rate': expected_return_rate,
            'property_value': property_value,
            'ltv_ratio': ltv_ratio,
            'annual_income': float(annual_income),
            'overdue_count': overdue_count,
            'employment_years': employment_years,
            'credit_score': credit_score,
            'debt_to_income_ratio': debt_to_income_ratio,
            'days_past_due_after_approval': 60 + risk_points * 25 if is_bad_debt else max(0, risk_points * 8),
            'cleared_principal_ratio': round(max(0.35, 1.0 - risk_points * 0.12), 2) if is_bad_debt else 1.0,
            'settlement_result': '坏账' if is_bad_debt else '正常清算',
            'is_bad_debt': is_bad_debt,
        })
    return records


clearance_records = build_clearance_records()
clearance_df = pd.DataFrame(clearance_records)
bad_debt_distribution_df = (
    clearance_df['settlement_result']
    .value_counts()
    .rename_axis('清算结果')
    .reset_index(name='样本数')
)

display(clearance_df.head(10))
display(bad_debt_distribution_df)


,loan_id,loan_amount,expected_return_rate,property_value,ltv_ratio,annual_income,overdue_count,employment_years,credit_score,debt_to_income_ratio,days_past_due_after_approval,cleared_principal_ratio,settlement_result,is_bad_debt
0,PL0001,220000.0,0.045,488888.89,0.45,120000.0,0,1,620,0.22,24,1.0,正常清算,0
1,PL0002,313000.0,0.048,626000.00,0.50,167000.0,0,4,637,0.29,24,1.0,正常清算,0
2,PL0003,406000.0,0.051,738181.82,0.55,214000.0,1,7,654,0.36,16,1.0,正常清算,0
3,PL0004,499000.0,0.054,831666.67,0.60,261000.0,2,10,671,0.43,24,1.0,正常清算,0
4,PL0005,592000.0,0.057,910769.23,0.65,308000.0,0,13,688,0.50,8,1.0,正常清算,0
5,PL0006,685000.0,0.060,978571.43,0.70,355000.0,3,1,705,0.57,24,1.0,正常清算,0
6,PL0007,778000.0,0.063,1037333.33,0.75,402000.0,1,4,722,0.64,16,1.0,正常清算,0
7,PL0008,871000.0,0.066,1088750.00,0.80,449000.0,0,7,739,0.26,8,1.0,正常清算,0
8,PL0009,964000.0,0.069,2142222.22,0.45,136000.0,2,10,756,0.33,24,1.0,正常清算,0
9,PL0010,1057000.0,0.072,2114000.00,0.50,183000.0,0,13,773,0.40,0,1.0,正常清算,0


,清算结果,样本数
0,正常清算,48
1,坏账,12


In [3]:
X = clearance_df.loc[:, BBB_KNN_VARIABLES]
y = clearance_df['is_bad_debt']

lightgbm_model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=80,
    learning_rate=0.06,
    num_leaves=7,
    min_data_in_leaf=3,
    min_data_in_bin=1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
lightgbm_model.fit(X, y)

gain_importance = lightgbm_model.booster_.feature_importance(importance_type='gain')
split_importance = lightgbm_model.booster_.feature_importance(importance_type='split')
feature_importance_df = pd.DataFrame({
    '变量': BBB_KNN_VARIABLES,
    'gain_importance': gain_importance,
    'split_importance': split_importance,
}).sort_values('gain_importance', ascending=False, kind='mergesort').reset_index(drop=True)

display(feature_importance_df)


,变量,gain_importance,split_importance
0,credit_score,146.118071,78
1,overdue_count,111.421859,103
2,expected_return_rate,74.084156,82
3,ltv_ratio,72.356842,50
4,loan_amount,31.770128,41
5,debt_to_income_ratio,27.777746,52
6,annual_income,23.415791,52
7,property_value,19.606720,18
8,employment_years,0.998741,4


In [4]:
def build_bbb_knn_weights(gain_values: np.ndarray, feature_names: list[str]) -> dict[str, float]:
    gain_series = pd.Series(gain_values.astype(float), index=feature_names)
    if gain_series.sum() <= 0:
        gain_series = pd.Series(1.0, index=feature_names)

    # 保留所有 BBB KNN 变量。极小兜底权重避免 0 权重让变量在欧式距离中完全失效。
    floor = gain_series[gain_series > 0].min() * 0.05 if (gain_series > 0).any() else 1.0
    adjusted = gain_series.clip(lower=floor)
    normalized = adjusted / adjusted.sum()
    return {name: float(normalized[name]) for name in feature_names}


bbb_knn_weights = build_bbb_knn_weights(gain_importance, BBB_KNN_VARIABLES)
bbb_knn_weights_df = pd.DataFrame({
    '变量': BBB_KNN_VARIABLES,
    '权重': [bbb_knn_weights[name] for name in BBB_KNN_VARIABLES],
}).assign(权重百分比=lambda df: (df['权重'] * 100).round(2).astype(str) + '%')

display(bbb_knn_weights_df)
print('bbb_knn_weights = {')
for name, weight in bbb_knn_weights.items():
    print(f"    '{name}': {weight:.10f},")
print('}')


,变量,权重,权重百分比
0,loan_amount,0.062595,6.26%
1,expected_return_rate,0.145964,14.6%
2,property_value,0.038630,3.86%
3,ltv_ratio,0.142561,14.26%
4,annual_income,0.046135,4.61%
5,overdue_count,0.219529,21.95%
6,employment_years,0.001968,0.2%
7,credit_score,0.287889,28.79%
8,debt_to_income_ratio,0.054729,5.47%


bbb_knn_weights = {
    'loan_amount': 0.0625950646,
    'expected_return_rate': 0.1459642363,
    'property_value': 0.0386301205,
    'ltv_ratio': 0.1425609967,
    'annual_income': 0.0461349400,
    'overdue_count': 0.2195288092,
    'employment_years': 0.0019677690,
    'credit_score': 0.2878889871,
    'debt_to_income_ratio': 0.0547290766,
}
